In [35]:
import networkx as nx
from networkx.algorithms.community import louvain_communities
from networkx.algorithms.community.quality import modularity
import json
import matplotlib.pyplot as plt
from operator import itemgetter
from typing import Set, Dict, Any, Tuple
import os

import graph_creation
from operator import itemgetter # Utilisé pour trier

In [36]:
# --- Constants for file paths ---
DEFAULT_JSON_PATH = 'data/processed/unified_articles.json'
FILTERED_JSON_PATH = 'data/processed/filtered_articles.json'

## 1. Function to filter the JSON based on in-degree and save the reduced version

def filter_json_and_save(
    json_path: str = DEFAULT_JSON_PATH, 
    output_json_path: str = FILTERED_JSON_PATH,
    top_n: int = 10000
) -> bool:
    """
    Reduces the JSON file by keeping only the articles corresponding to the 
    'top_n' non-isolated nodes with the highest in-degree in the full graph.
    Filters 'refs' to ensure no dangling pointers remain.
    """
    
    # 0. CHECK IF FILE ALREADY EXISTS
    if os.path.exists(output_json_path):
        print(f"✅ Filtered JSON file already exists at: {output_json_path}. Skipping filtering.")
        return True 
        
    print(f"Starting JSON filtering based on in-degree (top {top_n})...")
    
    # 1. Load JSON data
    try:
        with open(json_path, 'r', encoding='utf-8') as json_file:
            data: Dict[str, Any] = json.load(json_file)
            articles: list = data.get('articles', [])
            
    except Exception as e:
        print(f"Error loading JSON: {e}")
        return False

    # 2. Create the full graph
    print(f"Building initial full graph from {len(articles)} articles...")
    G_full = nx.DiGraph()
    article_id_to_data = {} 
    
    for article in articles:
        article_id = article.get('id')
        if article_id is not None:
            G_full.add_node(article_id) 
            article_id_to_data[article_id] = article
            for link in article.get('refs', []):
                G_full.add_edge(article_id, link)

    # 3. Select top N nodes
    print(f"Calculating in-degree and selecting top {top_n} nodes...")
    in_degrees = G_full.in_degree()
    sorted_nodes = sorted(in_degrees, key=itemgetter(1), reverse=True)
    
    # Set of IDs we want to keep
    final_node_ids: Set[Any] = {node for node, degree in sorted_nodes[:top_n]}

    # 4. (Optional) Graph filtering step logic
    # Note: Si vous voulez juste filtrer le JSON, l'étape 4 de création de G_temp 
    # n'est pas strictement nécessaire pour la sauvegarde, car nous avons final_node_ids.
    # Mais elle est utile si vous voulez recalculer des métriques ou supprimer les isolés.
    # Ici, on se base sur final_node_ids calculé ci-dessus.

    # 5. Create and save the filtered JSON with CLEANED refs
    print("Building filtered article list and cleaning references...")
    
    filtered_articles = []
    
    for node_id in final_node_ids:
        if node_id in article_id_to_data:
            # IMPORTANT: On fait une copie (shallow copy) pour ne pas modifier l'original en mémoire
            # si jamais on réutilise 'articles' plus tard dans le script.
            original_article = article_id_to_data[node_id]
            new_article = original_article.copy()
            
            # NETTOYAGE DES REFS :
            # On ne garde que les refs qui sont EGALEMENT dans final_node_ids
            original_refs = original_article.get('refs', [])
            cleaned_refs = [ref for ref in original_refs if ref in final_node_ids]
            
            new_article['refs'] = cleaned_refs
            filtered_articles.append(new_article)

    data_filtered = {"articles": filtered_articles}
    
    print(f"Filtered articles to be saved: {len(filtered_articles)}")
    
    try:
        os.makedirs(os.path.dirname(output_json_path) or '.', exist_ok=True)
        with open(output_json_path, 'w', encoding='utf-8') as f:
            json.dump(data_filtered, f, indent=4)
        print(f"Filtered JSON saved successfully to: {output_json_path}")
        return True
    except Exception as e:
        print(f"Error saving filtered JSON: {e}")
        return False

## 2. Function to build the graph from the filtered JSON (IN MEMORY ONLY)

def create_graph_from_filtered_json(
    json_path: str = FILTERED_JSON_PATH,
) -> nx.DiGraph:
    """
    Builds a directed graph (DiGraph) from the filtered JSON file.
    The graph is created in memory and is NOT saved or loaded from disk.
    """

    print("--- Creating graph from filtered JSON (in memory)... ---")
    
    G = nx.DiGraph()
    
    # 1. Load JSON data (Consolidated try/except)
    try:
        with open(json_path, 'r', encoding='utf-8') as json_file:
            data = json.load(json_file)
            
        articles = data.get('articles', [])
        
        # 2. Graph creation
        for article in articles:
            article_id = article.get('id')
            if article_id is not None:
                G.add_node(article_id) 
                for link in article.get('refs', []):
                    G.add_edge(article_id, link)
                    
             
    except FileNotFoundError:
        print(f"Error: Filtered JSON file not found at: {json_path}. Run 'filter_json_and_save' first.")
        return nx.DiGraph()
    except json.JSONDecodeError:
        print(f"Error: Invalid JSON format in file: {json_path}")
        return nx.DiGraph()
    except Exception as e:
        print(f"Error during graph creation: {e}")
        return nx.DiGraph()

    print(f"Graph created: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges.")
    
    return G

In [37]:
filter_json_and_save()

✅ Filtered JSON file already exists at: data/processed/filtered_articles.json. Skipping filtering.


True

## Plot graph

In [ ]:
G = create_graph_from_filtered_json()
# Calculation of the disposition (layout) - "Force Atlas"
print('Layout calculation')
try:
    pos = nx.forceatlas2_layout(G,gravity=70,max_iter=150,seed=110)
except Exception:
    pos = nx.spring_layout(G, seed=42)

print("Generation of the figure")
plt.figure(figsize=(15, 8))
nx.draw_networkx(
    G,
    pos,
    node_size=20,
    with_labels=False,
    width=0.05,
    edge_color="#706f6f"
)
plt.title(f"Graph Visualization (Force-Atlas Layout)", fontsize=20)

plt.axis('off')

print('Visualization')
plt.show()
plt.close()

--- Creating graph from filtered JSON (in memory)... ---
Graph created: 10000 nodes, 154970 edges.
Layout calculation


## Community generation

In [ ]:
def community_detection():
    results = [] 
    G = create_graph_from_filtered_json()


    # Louvain community detection
    communities = louvain_communities(G, seed=42)

    for comm in communities:
        # Identify the node with the highest degree as the representative
        representative_article = sorted(comm, key=lambda x: G.in_degree(x), reverse=True)[0]
        results.append({
            'representative_node' : representative_article,
            # Convert set to list for JSON serialization
            'community' : list(comm) 
        })

    with open('data/processed/communities.json', 'w') as outfile:
        # Use indent for better JSON readability
        json.dump(results, outfile, indent=4) 

if __name__ == '__main__':
    community_detection()

--- Creating graph from filtered JSON (in memory)... ---
Graph created: 10000 nodes, 154970 edges.


In [ ]:
file_path = 'data/processed/communities.json' 

with open(file_path, 'r') as infile:
    commu = json.load(infile)
print(len(commu))

article_name = {}
number_citation = {}
with open('data/processed/filtered_articles.json', 'r' ) as infile:
    nodes = json.load(infile)
for article in nodes['articles']:
    article_name[article['id']] = article['title']

87


In [ ]:
for i,commu_i in enumerate(commu):
    print(f"Commu numéro {i+1} : {article_name[commu_i['representative_node']]}")

Commu numéro 1 : Nonlinear Expectations and Stochastic Calculus under Uncertainty
Commu numéro 2 : Fashion-MNIST: a Novel Image Dataset for Benchmarking Machine Learning
  Algorithms
Commu numéro 3 : Bounds for the adiabatic approximation with applications to quantum
  computation
Commu numéro 4 : Flavor Structure of the Nucleon Sea from Lattice QCD
Commu numéro 5 : Controlled transfer of quantum amplitude via modulation of a potential
  barrier: numerical study in a model of SQUID
Commu numéro 6 : On the Geometry and Homology of Certain Simple Stratified Varieties
Commu numéro 7 : The Large N Limit of Superconformal Field Theories and Supergravity
Commu numéro 8 : Gaia Data Release 2: The astrometric solution
Commu numéro 9 : Fermion-boson duality in integrable quantum field theory
Commu numéro 10 : Entanglement Entropy of Systems with Spontaneously Broken Continuous
  Symmetry
Commu numéro 11 : String Topology
Commu numéro 12 : What is the gamma gamma resonance at 750 GeV?
Commu numé

## Recommandation

In [ ]:
# Core Python libraries
import numpy as np
import pandas as pd
from tqdm import tqdm
import json
import re
import os

# NLP and Embeddings
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity
import faiss


# Visualization and evaluation
import matplotlib.pyplot as plt
import seaborn as sns



In [ ]:
def compute_and_save_embeddings(texts, doc_ids, model_name='all-mpnet-base-v2',
                                batch_size=64, out_emb_path='embeddings.npy',
                                out_ids_path='doc_ids.json', overwrite=True):
    if os.path.exists(out_emb_path) and not overwrite:
        raise FileExistsError(f"{out_emb_path} already exists. Set overwrite=True to replace.")

    model = SentenceTransformer(model_name)
    n = len(texts)
    emb_dim = model.get_sentence_embedding_dimension()

    emb_memmap = np.lib.format.open_memmap(out_emb_path, mode='w+', dtype='float32', shape=(n, emb_dim))

    for i in tqdm(range(0, n, batch_size), desc="Embedding batches"):
        batch_texts = texts[i:i+batch_size]
        batch_emb = model.encode(batch_texts, show_progress_bar=False, convert_to_numpy=True)
        # normalize rows to unit vectors (for cosine via inner product)
        norms = np.linalg.norm(batch_emb, axis=1, keepdims=True)
        norms[norms == 0] = 1.0
        batch_emb = batch_emb / norms
        emb_memmap[i:i+len(batch_emb)] = batch_emb.astype('float32')

    # ensure data flushed to disk
    del emb_memmap

    # save doc ids as JSON
    with open(out_ids_path, 'w', encoding='utf-8') as f:
        json.dump(list(doc_ids), f, ensure_ascii=False)

    print(f"Saved embeddings -> {out_emb_path}")
    print(f"Saved doc ids -> {out_ids_path}")

In [ ]:
with open('data/processed/filtered_articles.json', 'r', encoding='utf-8') as f:
    data = json.load(f)


texts = [article["clean_text"] for article in data["articles"]]
doc_ids = [article["id"] for article in data["articles"]]


model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Appel de la fonction
compute_and_save_embeddings(
    texts,
    doc_ids,
    model_name='all-MiniLM-L6-v2',
    batch_size=8,
    out_emb_path='data/processed/embeddings.npy',
    out_ids_path='data/processed/doc_ids.json',
    overwrite=True
)

# Chargement et vérification
loaded_emb = np.load('data/processed/embeddings.npy', mmap_mode='r')
with open('doc_ids.json', 'r', encoding='utf-8') as f:
    loaded_ids = json.load(f)

print("Embeddings shape:", loaded_emb.shape)
print("Number of doc ids:", len(loaded_ids))
print("Example embedding (first doc) first 10 dims:", loaded_emb[0][:10])

c:\Users\tchir\miniforge3\envs\comp_env\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\tchir\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling b

Saved embeddings -> embeddings.npy
Saved doc ids -> doc_ids.json
Embeddings shape: (10000, 384)
Number of doc ids: 10000
Example embedding (first doc) first 10 dims: [-0.10747099  0.02192722 -0.02454343  0.00243487 -0.03293996  0.08883931
  0.02780451 -0.06044067  0.02010144 -0.05245653]


In [ ]:
def build_faiss_index(embeddings_path='data/processed/embeddings.npy', index_path='data/processed/faiss.index',
                      index_type='hnsw', ef_construction=200, M=32):
    emb = np.load(embeddings_path, mmap_mode='r')  # shape (N, d)
    d = emb.shape[1]
    if index_type == 'flat':
        index = faiss.IndexFlatIP(d)  # inner product -> cosine if vectors normalized
        index.add(emb)
    elif index_type == 'hnsw':
        index = faiss.IndexHNSWFlat(d, M)  # M controls connectivity
        index.hnsw.efConstruction = ef_construction
        index.add(emb)
    else:
        raise ValueError('index_type not supported')
    faiss.write_index(index, index_path)
    return index

In [ ]:
def retrieve_similar_articles(query, model, embeddings, articles, top_n=5, use_ann=False):
    query_emb = model.encode([query], convert_to_numpy=True)
    # normalize
    query_emb = query_emb / np.linalg.norm(query_emb, axis=1, keepdims=True)
    
    if use_ann:
        index = build_faiss_index()
        distances, indices = index.search(query_emb.astype('float32'), top_n)
    else:
        # Exact search using sklearn
        nbrs = NearestNeighbors(n_neighbors=top_n, metric="cosine").fit(embeddings)
        distances, indices = nbrs.kneighbors(query_emb)
    
    results = pd.DataFrame({
        "article_id": [articles[i] for i in indices[0]],
        "similarity": [1 - d for d in distances[0]]
    })
    return results

In [ ]:
def get_representative(member_id, file_path="data/processed/communities.json"):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        for group in data:
            if member_id in group.get("community", []):
                return group["representative_node"]
                
        return None

    except (FileNotFoundError, json.JSONDecodeError):
        return None

In [ ]:
query  ="Deputee repartition"


top_n = 1

embeddings = np.load('embeddings.npy', mmap_mode='r')
results = retrieve_similar_articles(query, model, embeddings, doc_ids, top_n, use_ann=True)

print('QUERY :')
print(query)
print("---")

print(f"Closest article")
print(article_name[results['article_id'][0]])
print("----")

print("Represenrative article")
representative = get_representative(results['article_id'][0])
print(article_name[representative])


QUERY :
Deputee repartition
---
Closest article
The automated computation of tree-level and next-to-leading order
  differential cross sections, and their matching to parton shower simulations
Number of citation : 147
----
Represenrative article
Bounds for the adiabatic approximation with applications to quantum
  computation
Number of citation : 21
